# Datamine NSCORE Process Example

This notebook demonstrates how to configure and run the **`nscore`** process wrapper in `dmstudio`.

## Process Description

## NSCORE

**Note** : This is a _superprocess_ and running it may have an effect on other Datamine files in the project.

Transform a set of data to a normal distribution.

* **Note** (This process is based on the GSLIB code for normal score transformation (nscore). See the footnotes for accreditation and source.):

If a **REFDIST** file is not specified, then **NSCORE** transforms the data to a normal distribution using the standard Gaussian cumulative density function method. If a **REFDIST** file is specified then **NSCORE** transforms the data to a distribution defined by **REFDIST**.

The SGSIM process includes an option that carries out a normal transformation prior to the simulation. However it is useful to have the separate process **NSCORE** to do just the transformation, because the input variogram model for SGSIM should be calculated from the normal data.

### Program Limits

**Maximum number of samples** in input IN file 500000

The following limitations currently apply to the **folder name of the project** :

* A maximum of 48 characters in the full path name of the project folder, for example c:\database\consim\project1 must not exceed 80 characters.

* The full path name of the project folder must not include spaces, for example c:\database\consim\my project1 is not allowed.

### System Files

The following temporary and system files are generated by the NSCORE process. In typical use, these are unimportant and cleared up automatically.

* _nsclog.txt Log file. Only useful if there is a problem.

* _nsc_*.txt Temporary system files. These will be deleted if the process terminates cleanly.

* _sp*.dm Temporary Datamine files. These will be deleted if the process terminates cleanly.

All files matching the template _nsc_*.txt and _sp*.dm will be deleted as the process terminates. Therefore you should not use any of these file names.

### Input Files:

* **in** (Table):
  Input sample data file. This must include the grade field **GRADE** and may also
  include the declustering weight field **DCWGT** .
  Required=Yes

* **refdist** (Table):
  Optional input reference distribution to define required transformation. The file must
  include the field **REFGRADE** , to define the distribution, and may also include the
  field **REFWGT** to define declustering weights.
  Required=No

### Output Files:

* **out** (Block Model):
  Output file containing transformed points. This contains the same data as the IN file,
  but with the added transformed data field **NSCORE**.
  Required=Yes

* **trandist** (Table):
  Output file for the transformation table. This will contain the grade field **GRADE**
  from the IN sample file and the field **TRANDATA** giving the transformed value. The
  file will be sorted by **GRADE** .
  Required=No

* **stat_tbl** (Table):
  Output statistics table. This provides statistics for both the input, untransformed,
  sample data, as well as the output, transformed, values.
  Required=No

### Fields:

* **grade** (Numeric : IN):
  Field in the input sample file **IN** defining the grade to be transformed.
  Default=Undefined
  Required=Yes

* **dcwgt** (Numeric : IN):
  Optional declustering weights field in the **IN** file.
  Default=Undefined
  Required=No

* **refgrade** (Numeric : REFDIST):
  Reference grade field, defining the reference distribution in the **REFDIST** file.
  Default=Undefined
  Required=No

* **refwgt** (Numeric : REFDIST):
  Optional reference weight field, defining declustering weights in the **REFDIST** file.
  Default=Undefined
  Required=No

### Parameters:

* **mingrade**:
  Minimum **GRADE** value input from the IN sample file.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **maxgrade**:
  Maximum **GRADE** value input from the **IN** sample file.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('nscore')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.initialize_sandbox(notebook_folder, files_to_copy=files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute nscore
print("Running nscore...")
dm_cmd.nscore(
    in_i='t_assays',  # required
    refdist_i='optional',  # required
    out_o='t_nscore_out',  # required
    stat_tbl_o='t_nscore_out',  # required
    grade_f='optional',  # required
    # trandist_o='t_nscore_out',  # optional
    # dcwgt_f='optional',  # optional
    # refgrade_f='optional',  # optional
    # refwgt_f='optional',  # optional
    # mingrade_p=0,  # optional
    # maxgrade_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("nscore execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_nscore_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")